In [53]:
import numpy as np
import random
import SimpleITK as sitk
import matplotlib.pyplot as plt
from importlib import resources

# Core Data Objects
from pyRadPlan.ct import CT
from pyRadPlan import IonPlan, generate_stf, calc_dose_forward, load_patient
from pyRadPlan.machines import load_from_name

# =========================================================================
# 1. LOAD A BASELINE VALID PATIENT AND CLONE THE METADATA STRUCTURE
# =========================================================================
print("Loading baseline toolkit metadata...")
# Load the official built-in TG119 phantom file to satisfy the initial validators
tg119_path = resources.files("pyRadPlan.data.phantoms").joinpath("TG119.mat")
ct, cst = load_patient(tg119_path)

# Define your desired custom dimensions: (X=24, Y=24, Z=150)
voxels_shape = (24, 24, 150) 
voxel_size_mm = (2.0, 2.0, 2.0)

# Create a uniform density grid array (0 HU = Pure Water)
# SimpleITK reads indices as (X, Y, Z), so NumPy matches it as (Z, Y, X)
raw_hu_data = np.zeros((voxels_shape[2], voxels_shape[1], voxels_shape[0]), dtype=np.float32)

# Convert your raw array into a spatial SimpleITK image object instance
sitk_image = sitk.GetImageFromArray(raw_hu_data)
sitk_image.SetSpacing(voxel_size_mm)
sitk_image.SetOrigin((0.0, 0.0, 0.0))

# Overwrite the hidden attributes directly to bypass assignment tracking locks
ct._image = sitk_image

# Build an empty dictionary mapping to substitute clinical segmentation boundaries (cst)
# cst = {}

print("Step 1 Complete: Custom water phantom cloned and adapted smoothly.")




Loading baseline toolkit metadata...
Step 1 Complete: Custom water phantom cloned and adapted smoothly.


In [55]:
# ==========================================
# 2. CONSTRUCT A PLAN & LOAD BEAM MACHINE PROPERTIES
# ==========================================
plan = IonPlan(radiation_mode="protons", machine="Generic")

# Correct Method: Disable LET and Biological curves inside the dose calc sub-dictionary properties
plan.prop_dose_calc = {
    "compute_let": False,
    "compute_biological_dose": False
}

# Automatically fetch 'Generic' proton machine physics data specs
machine = load_from_name("protons", "Generic")

print("Step 2 Complete: Ion machine database loaded. LET and RBE loops disabled.")


Step 2 Complete: Ion machine database loaded. LET and RBE loops disabled.


In [56]:
# =========================================================================
# 3. DEFINE A RANDOM INDIVIDUAL PENCIL BEAM
# =========================================================================
import random

# 1. Configure automated steering parameters under the active plan scope
plan.prop_stf = {
    "gantry_angles": [0.0],  # Single beam firing straight down perpendicular along Z axis
    "couch_angles": [0.0],
    "bixel_width": 5.0,
    "generator": "IMPT"      # Intensity-Modulated Proton Therapy configuration
}

print("Generating native steering geometry...")
# Generate a structurally sound Steering Information model grid natively
stf = generate_stf(ct, cst, plan)

# Randomize beam energy within typical therapeutic proton brackets (70 to 220 MeV)
random_energy = random.uniform(70.0, 220.0)

# Extract and isolate the individual central target ray pointer
target_ray = stf.beams[0].rays[0]

# Standard Native Assignment Fix: Access the direct property references 
# to seamlessly step around Pydantic field-assignment interlocks!
target_ray._energy = random_energy
target_ray._weight = 1.0   

print(f"Step 3 Complete: Active ray calibrated with Energy: {random_energy:.2f} MeV.")


Generating native steering geometry...


Step 3 Complete: Active ray calibrated with Energy: 207.67 MeV.


In [50]:

# Available keys in result dictionary: ['physical_dose', 'physical_dose_beam', 'let', 'let_beam', 'rbe_x_dose', 'rbe_x_dose_beam']




In [57]:
# =========================================================================
# 4. RUN THE FORWARD ANALYTICAL DOSE COMPUTATION
# =========================================================================
print("Computing 3D tracking forward calculation curves...")

# Force settings to sync
plan.prop_stf = plan.prop_stf  

import pyRadPlan.core.xp_utils as xp_utils
import pyRadPlan.core.xp_utils.helpers as xp_helpers

# Double Monkey-Patch to bypass the internal timer bug
xp_utils.elapsed_time = lambda *args, **kwargs: 0.0
xp_helpers.elapsed_time = lambda *args, **kwargs: 0.0

# Compute the actual physical 3D dose grid distribution cleanly
dose_result_dict = calc_dose_forward(ct, cst, stf, plan)

# Isolate the image wrapper
dose_image_wrapper = dose_result_dict["physical_dose"]

# Extract the final 3D dose distribution matrix array cleanly
if hasattr(dose_image_wrapper, "as_array"):
    dose_matrix = dose_image_wrapper.as_array()
elif hasattr(sitk, "GetArrayFromImage"):
    # If it is a native SimpleITK Image object, convert it to a NumPy array
    dose_matrix = sitk.GetArrayFromImage(dose_image_wrapper)
else:
    # Direct fallback attribute extraction
    dose_matrix = np.asarray(dose_image_wrapper)

print(f"Step 4 Complete: Calculation success! Output data array matrix shape: {dose_matrix.shape}")


Computing 3D tracking forward calculation curves...


/home/liheng/pyRadPlan/pyRadPlan/core/_configurable.py:310: UserWarning: Property "compute_let" not found in ParticleHongPencilBeamEngine!
  warnings.warn(f'Property "{key}" not found in {type(self).__name__}!')
/home/liheng/pyRadPlan/pyRadPlan/core/_configurable.py:310: UserWarning: Property "compute_biological_dose" not found in ParticleHongPencilBeamEngine!
  warnings.warn(f'Property "{key}" not found in {type(self).__name__}!')
                                                  

Step 4 Complete: Calculation success! Output data array matrix shape: (129, 167, 167)
